# 05 · Chat Sessions & Operator Audit
Explore the chat session lifecycle, probe the streaming endpoint contract,
and pull operator audit timeline data.

**Covered endpoints**
- `POST /chat` — start or continue a chat session
- `GET  /chat/{sessionId}` — session detail
- `GET  /chat` — session list
- `DELETE /chat/{sessionId}` — close a session
- `GET  /streaming/status` — streaming endpoint health
- `GET  /audit` — operator audit log
- `GET  /telemetry` — platform telemetry snapshot

In [ ]:
import json, urllib.request, urllib.error
API_BASE = 'https://api.banana.engineer'

def pp(obj):
    if isinstance(obj, str):
        try: obj = json.loads(obj)
        except Exception: print(obj); return
    print(json.dumps(obj, indent=2))

def call_endpoint(method, path, payload=None, headers=None):
    url = f"{API_BASE}{path}"
    h = {'Accept': 'application/json'}
    if headers: h.update(headers)
    data = None
    if payload is not None:
        h['Content-Type'] = 'application/json'
        data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers=h, method=method.upper())
    try:
        with urllib.request.urlopen(req) as r:
            return {'ok': True, 'status': r.status, 'body': r.read().decode('utf-8', errors='replace')}
    except urllib.error.HTTPError as e:
        return {'ok': False, 'status': e.code, 'body': e.read().decode('utf-8', errors='replace')}
    except Exception as ex:
        return {'ok': False, 'status': None, 'body': str(ex)}

def get(path, **kw):    return call_endpoint('GET',    path, **kw)
def post(path, **kw):   return call_endpoint('POST',   path, **kw)
def delete(path, **kw): return call_endpoint('DELETE', path, **kw)
print('setup_ok')

In [ ]:
# ── 1. List chat sessions ─────────────────────────────────────────────────
r = get('/chat')
print('status:', r['status'])
pp(r['body'])

try:
    sessions = json.loads(r['body'])
    session_ids = [
        s.get('id') or s.get('sessionId')
        for s in (sessions if isinstance(sessions, list) else [])
    ]
    print('session_count:', len(session_ids))
except Exception:
    session_ids = []

In [ ]:
# ── 2. Start a new chat session ───────────────────────────────────────────
r = post('/chat', payload={'message': 'hello from banana DS lab'})
print('status:', r['status'])
pp(r['body'])

try:
    created = json.loads(r['body'])
    new_session_id = created.get('id') or created.get('sessionId')
    print('new_session_id:', new_session_id)
except Exception:
    new_session_id = None

In [ ]:
# ── 3. Session detail ─────────────────────────────────────────────────────
probe_id = new_session_id or (session_ids[0] if session_ids else None)
if probe_id:
    r = get(f'/chat/{probe_id}')
    print('status:', r['status'])
    pp(r['body'])
else:
    print('no_session_id_available')

In [ ]:
# ── 4. Streaming endpoint health ──────────────────────────────────────────
r = get('/streaming/status')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 5. Operator audit log ─────────────────────────────────────────────────
r = get('/audit')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 6. Telemetry snapshot ─────────────────────────────────────────────────
r = get('/telemetry')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 7. Session lifecycle summary ──────────────────────────────────────────
# Re-list sessions to confirm the newly created one appears
r2 = get('/chat')
try:
    sessions_after = json.loads(r2['body'])
    count_after = len(sessions_after) if isinstance(sessions_after, list) else '?'
except Exception:
    count_after = '?'

print(f'sessions_before: {len(session_ids)}  sessions_after: {count_after}')